# PHASE 3–4 — Original Baseline Modeling

This notebook contains the project's original feature engineering and baseline
interpolation work.

**Important:** the original random 10% masking experiment is historical/
exploratory and is **not** the official FinanceMeta evaluation.

The official leakage-safe benchmark is:

`notebooks/02_reproducible_benchmark.ipynb`

See `docs/protocol.md` for the frozen split, matched/nested holdout,
missingness levels, models, metrics, and no-retuning policy.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error, mean_absolute_error

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = REPO_ROOT / "data" / "dataset.csv"

df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"], format="%d-%m-%Y %H:%M", errors="raise")

option_cols = [c for c in df.columns if c.startswith("NIFTY")]

df_long = df.melt(
    id_vars=["datetime", "underlying_price"],
    value_vars=option_cols,
    var_name="symbol",
    value_name="iv",
)

df_long["expiry"] = pd.to_datetime(
    df_long["symbol"].str[5:12],
    format="%d%b%y",
    errors="raise",
)
df_long["strike"] = df_long["symbol"].str[12:-2].astype(float)
df_long["opt_type"] = df_long["symbol"].str[-2:]
df_long["moneyness"] = df_long["strike"] / df_long["underlying_price"]
df_long["log_moneyness"] = np.log(df_long["moneyness"])

print("Canonical long-form rows:", len(df_long))

## Historical strike-wise interpolation

The original project used linear interpolation over strike and compared it
with time-wise interpolation. Those experiments established strike-wise
interpolation as a useful baseline.

For the FinanceMeta trial, the implementation has been moved into
`src/interpolation.py` and the official benchmark uses a frozen structured
holdout with no silent extrapolation.

## Why this notebook is not the final evidence

The original random masking code is retained only as project history.

It should not be used to claim leakage-safe final performance because the
FinanceMeta checkpoint requires:

- a frozen protocol
- matched/nested 10/20/30% holdouts
- at least two simple baselines
- SVI comparison
- RMSE + MAE
- runtime + failure reporting
- consistency checks
- robustness testing
- negative-result preservation

Those requirements are implemented in Notebook 02.

## Original reconstruction/export work

The existing `src/export_submission.py` remains a separate legacy utility for
the original project workflow. It is not part of the FinanceMeta benchmark and
does not modify the raw `data/dataset.csv` source.